## Node-style hooks函数用法
支持两种用法
- 装饰器是函数式挂载，把一个hook快速挂载到Agent的某个节点。
- 类写法是对象化中间件，把中间件封装为一个可配置、可复用、可扩展的组件。

In [1]:
# 基于装饰器实现
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [ ]:
from langchain.agents.middleware import (
    before_model,
    after_model,
    before_agent,
    after_agent,
    AgentState,
    AgentMiddleware,
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


# 1. 定义 before_model 钩子
@before_model
def before_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None


# 2. 定义 after_model 钩子
@after_model
def after_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None


# 3. 定义 before_agent 钩子
@before_agent
def before_agent_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None


# 4. 定义 after_agent 钩子
@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    state["messages"][-1].content += " -> after_agent <- "
    return None


agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ],  # 👈 添加中间件
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:

    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！已收到你的输入流标识：`你好啊 -> before_agent <-  -> before_model <-`。

在日常交互中，这通常表示数据会先经过 `before_agent`（如权限校验、路由、日志记录等），再进入 `before_model`（如 Prompt 组装、上下文裁剪、安全过滤等），最后才由大模型生成回复。

如果你是在调试自定义 Hook、中间件或工作流框架，我可以：
- 帮你梳理各阶段的输入/输出格式
- 提供对应代码片段（Python/JS/Node.js 等）
- 模拟后续 `model_output -> after_model <-` 的完整流程

需要我侧重哪一部分？随时告诉我～ 😊 -> after_model <-  -> after_agent <-


# 基于类实现
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

LangGraph 只看：

是否继承 AgentMiddleware？

是否有 before_model / after_model 等方法？

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()

    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_model <- "
        return None

    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None

    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        state["messages"][-1].content += " -> after_agent <- "
        return None


my_middleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[my_middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:
    msg.pretty_print()

1. before_model通用场景：
- 消息修剪（trim messages）
- PII 脱敏
- 输入验证
- 条件路由

2.  after_model 通常的场景：
- 输出验证
- 格式化响应
- 统计信息
- 状态更新

### 2种方法的统一
装饰器底层会基于我们重写的方法构造一个AgentMiddleware子类的实例，以@after_model为例：

In [ ]:
# 这是after_model最终返回的内容
return type(
    middleware_name,
    (AgentMiddleware,),
    {
        "state_schema": state_schema or AgentState,
        "tools": tools or [],
        "after_model": wrapped,
},
)()